# Feature Engineering

## Objective

The objective of this notebook is to prepare the dataset for machine learning model development by applying preprocessing and feature engineering techniques.

The transformed dataset will be used for model experimentation and evaluation.

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Dell/OneDrive/Desktop/mlops-LoanPrediction-Poject/data/raw/loan_data.csv")
df.head(5)

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [9]:
df_fe = df.copy()

A copy of the dataset is created to preserve the original data and ensure reproducibility.

## Feature Creation

### Credit History per Age

In [10]:
df_fe["credit_history_ratio"] = (
    df_fe["cb_person_cred_hist_length"]
    /
    df_fe["person_age"] )

Measures how much credit history a person has relative to their age.

### Income per Year of Age

In [11]:
df_fe["income_per_age"] = (
    df_fe["person_income"]
    /
    df_fe["person_age"]
)

Normalizes income based on age.

## Spliting Data

In [12]:
X = df_fe.drop("loan_status",axis=1)
y = df_fe["loan_status"]

The target variable is separated from the predictor variables to prepare for model training.

### Handling Multicollinearity

In [13]:
df_fe[["person_age","person_emp_exp"]].corr()

,person_age,person_emp_exp
person_age,1.000000,0.954412
person_emp_exp,0.954412,1.000000


A strong correlation was observed between age and employment experience.

To reduce redundancy and improve model interpretability, `person_emp_exp` was removed.

In [14]:
X = X.drop("person_emp_exp", axis=1)

### Identify Feature Type

In [15]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

print(cat_cols)
print(num_cols)

Index(['person_gender', 'person_education', 'person_home_ownership',
       'loan_intent', 'previous_loan_defaults_on_file'],
      dtype='object')
Index(['person_age', 'person_income', 'loan_amnt', 'loan_int_rate',
       'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score',
       'credit_history_ratio', 'income_per_age'],
      dtype='object')


### Encoding Categorical Variables

In [16]:
X["person_gender"] = X["person_gender"].map({
    "male": 1,
    "female": 0
})

X["previous_loan_defaults_on_file"] = X["previous_loan_defaults_on_file"].map({
    "Yes": 1,
    "No": 0
})


cat_col = [
    "person_education",
    "person_home_ownership",
    "loan_intent"
]

X = pd.get_dummies(
    X,
    columns=cat_col,
    drop_first=True,
    dtype=int
)
X.head(4)

,person_age,person_gender,person_income,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,credit_history_ratio,...,person_education_High School,person_education_Master,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22.0,0,71948.0,35000.0,16.02,0.49,3.0,561,0,0.136364,...,0,1,0,0,1,0,0,0,1,0
1,21.0,0,12282.0,1000.0,11.14,0.08,2.0,504,1,0.095238,...,1,0,0,1,0,1,0,0,0,0
2,25.0,0,12438.0,5500.0,12.87,0.44,3.0,635,0,0.120000,...,1,0,0,0,0,0,0,1,0,0
3,23.0,0,79753.0,35000.0,15.23,0.44,2.0,675,0,0.086957,...,0,0,0,0,1,0,0,1,0,0


### Train-Test Split

In [17]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify = y
)


The dataset was divided into training and testing subsets using an 80:20 ratio while preserving class distribution.

### Scaling

In [18]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    "person_age",
    "person_income",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "credit_score",
    "credit_history_ratio",
    "income_per_age"
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(
    X_train[num_cols]
)
X_test_scaled[num_cols] = scaler.transform(
    X_test[num_cols]
)

Numerical variables were standardized using StandardScaler.
Scaling is particularly beneficial for algorithms such as Logistic Regression that are sensitive to feature magnitude.

### Save Processed Datasets

In [19]:
X_train_scaled.to_csv(
    "C:/Users/Dell/OneDrive/Desktop/mlops-LoanPrediction-Poject/data/processed/X_train.csv",
    index=False
)

X_test_scaled.to_csv(
    "C:/Users/Dell/OneDrive/Desktop/mlops-LoanPrediction-Poject/data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "C:/Users/Dell/OneDrive/Desktop/mlops-LoanPrediction-Poject/data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "C:/Users/Dell/OneDrive/Desktop/mlops-LoanPrediction-Poject/data/processed/y_test.csv",
    index=False
)

# Feature Engineering Summary

The following preprocessing steps were completed:

- Additional features were engineered to capture applicant financial and credit characteristics.
- Removed highly correlated features
- Applied binary encoding
- Applied one-hot encoding
- Split the dataset into training and testing sets
- Standardized numerical features
- Saved processed datasets and scaler

The data is now ready for model training and experimentation.